## 3.1 — Cargar documentos y convertirlos al formato de LangChain

**Qué se hace:** se reutiliza `cargar_carpeta()` (Fase 1) para leer los 3
documentos ficticios, y se convierten con `documentos_a_langchain()` al
formato `Document` que espera LangChain (`page_content` + `metadata`).

**Para qué:** todo lo que viene después en esta fase (splitter, embeddings,
ChromaDB) trabaja con objetos `Document`, no con nuestro formato propio
`DocumentoCargado`. Este paso es el "traductor" entre ambos mundos.

**En qué fijarse en el resultado:**
- Que aparezcan los 3 documentos, ni de más ni de menos.
- Que la `metadata` de cada uno tenga el `tipo` correcto (`financiero`,
  `convenio`, `agencia_colocacion`) — esto será clave más adelante para que
  el Agente Redactor (Fase 4) solo use el contexto del tipo de documento
  correcto al redactar cada sección.
- Las longitudes (240-758 caracteres) confirman que los documentos
  ficticios son muy pequeños — dato importante para interpretar el
  resultado del siguiente paso (el splitter).

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.rag.loader import cargar_carpeta
from src.rag.adaptador_langchain import documentos_a_langchain

documentos = cargar_carpeta(Path.cwd().parent / "data" / "raw")
documentos_lc = documentos_a_langchain(documentos)

for doc in documentos_lc:
    print(doc.metadata)
    print(f"Longitud: {len(doc.page_content)} caracteres\n")

## 3.2 — Dividir los documentos en chunks

**Qué se hace:** se usa `RecursiveCharacterTextSplitter` para dividir cada
`Document` en fragmentos más pequeños (`chunk_size=500`, con solapamiento
de 50 caracteres entre chunks consecutivos).

**Para qué:** cuando lleguen documentos reales (más largos que los
ficticios), no se puede pasar el documento completo al modelo en cada
consulta — el RAG necesita indexar fragmentos pequeños para poder
recuperar solo la parte relevante de cada sección de la memoria.

**Resultado obtenido: 3 documentos → 5 chunks**

| Documento | Chunks | Por qué |
|---|---|---|
| `agencia_colocacion.xlsx` (240 caracteres) | 1 | Más pequeño que `chunk_size=500`; no hay nada que dividir |
| `convenios_2026.docx` (743 caracteres) | 2 | Se divide aproximadamente donde empieza cada convenio, gracias al separador `\n\n` (párrafo), que respeta los títulos `## Convenio con...` |
| `financiero_cuatrimestral_1.pdf` (758 caracteres) | 2 | Se divide separando el texto narrativo (resumen, indicadores) de la tabla de partidas — casualmente coincide con la misma frontera que ya trazamos a propósito en el `loader.py` (Fase 1) al excluir el área de las tablas del texto narrativo |

**En qué fijarse (importante, no solo para hoy sino cada vez que se
reajuste el splitter más adelante):**
- **Que ningún dato quede cortado a la mitad.** Por ejemplo, revisar que
  una línea tipo `Partida: Formacion y empleo | Presupuesto: 850.000 EUR |
  Ejecutado: ...` no termine partida en un chunk y continúe en el
  siguiente — si eso pasara, el sistema de recuperación podría devolver
  un dato incompleto o sin su contexto (imaginemos recuperar el `Presupuesto`
  de una partida sin el `Ejecutado` correspondiente).
- **Que el chunk de `convenios_2026.docx` muestre solo 2 fragmentos pero
  haya 3 convenios en el documento (Cáritas, Cámara de Comercio, Cruz
  Roja).** Esto indica que uno de los chunks contiene más de un convenio
  junto — no es un error, pero conviene tenerlo presente: si más adelante
  se recupera ese chunk para "hablar de Cáritas", vendrá acompañado de
  datos de otro convenio distinto en el mismo fragmento.
- **Que los documentos pequeños (como el Excel, 240 caracteres) generen
  solo 1 chunk.** Es el comportamiento esperado, no un fallo — confirma
  que el splitter no fragmenta innecesariamente contenido que ya es
  pequeño.

**Limitación conocida de esta prueba:** con documentos ficticios tan
pequeños, el splitter apenas tiene margen para mostrar su comportamiento
real. La validación definitiva de `chunk_size` y `chunk_overlap` (tarea
3.7) deberá repetirse cuando lleguen documentos reales del Ayuntamiento,
presumiblemente más largos.

## Añadimos TxtSplitter con chunks de 500 y overlap 50 pensado para los datos reales

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(documentos_lc)

print(f"Documentos originales: {len(documentos_lc)}")
print(f"Chunks generados: {len(chunks)}\n")

for chunk in chunks:
    print(f"--- {chunk.metadata['fuente']} ---")
    print(chunk.page_content[:150])
    print("...\n")

## 3.2b — Revisión manual completa de los chunks

**Qué se hace:** se imprime el contenido completo (no solo los primeros
150 caracteres) de cada chunk generado, para confirmar visualmente que
ningún dato quedó cortado de forma problemática — en particular, el
chunk de convenios que mezcla varios convenios juntos, y las partidas
financieras, donde cortar a mitad de una línea "Partida: X | Presupuesto:
Y | Ejecutado: Z" perdería la relación entre esos datos.

**Por qué se hace ahora y no más adelante:** es mucho más barato detectar
un problema de chunking ahora, revisando texto plano, que después de
haber construido embeddings y la base vectorial sobre datos ya mal
fragmentados — habría que rehacer todo ese trabajo.

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"=== Chunk {i} — {chunk.metadata['fuente']} ({len(chunk.page_content)} caracteres) ===")
    print(chunk.page_content)
    print()

## Hallazgo: solapamiento imperfecto en el corte de convenios

Al revisar el contenido completo de los chunks, se observa que el
`chunk_overlap=50` provoca que el título "## Convenio con la Camara de
Comercio" aparezca duplicado: una vez al final del Chunk 1 (sin su
contenido) y otra vez al principio del Chunk 2 (con su contenido
completo). No hay pérdida de datos, pero es una señal de que el
`chunk_size=500` con estos datos concretos corta justo en un punto poco
ideal.

**Decisión:** no se ajusta ahora mismo, porque los documentos ficticios
son demasiado pequeños para sacar una conclusión fiable sobre el tamaño
óptimo de chunk (ver limitación ya anotada en 3.2). Se deja registrado
para revisar en la tarea 3.7 (ajuste de chunk_size/overlap) cuando se
disponga de documentos reales, más representativos del caso de uso real.

## 3.3 — Generar embeddings de un chunk (prueba individual)

**Qué se hace:** se genera el embedding (vector numérico) del primer chunk,
usando `nomic-embed-text` a través de Ollama, para confirmar que la
conexión funciona y entender qué produce este paso antes de aplicarlo a
todos los chunks.

**Para qué:** los embeddings son la base de la búsqueda semántica del RAG
— permiten comparar "qué tan parecido en significado" es un fragmento de
texto respecto a una pregunta, sin depender de que compartan las mismas
palabras exactas.

**En qué fijarse:** el número de dimensiones del vector debe ser siempre
el mismo para cualquier texto que se le pase a este modelo (normalmente
768 con `nomic-embed-text`) — si cambiara entre ejecuciones, indicaría un
problema de configuración del modelo.

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings_modelo = OllamaEmbeddings(model="nomic-embed-text")

vector_ejemplo = embeddings_modelo.embed_query(chunks[0].page_content)

print(f"Dimensiones del vector: {len(vector_ejemplo)}")
print(f"Primeros 10 valores: {vector_ejemplo[:10]}")

## 3.4 — Crear e indexar la base de datos vectorial (ChromaDB)

**Qué se hace:** se indexan los 5 chunks en ChromaDB usando
`Chroma.from_documents()`, que genera automáticamente el embedding de
cada chunk (con `nomic-embed-text`, ya probado en el paso 3.3) y lo
guarda en una base de datos vectorial persistida en disco
(`data/chroma_db/`), bajo la colección `memoria_ayuntamiento`.

**Para qué:** esta es la pieza central del RAG — a partir de ahora, en
vez de tener que pasarle al modelo todo el texto de todos los documentos
en cada consulta, se puede preguntar "¿qué fragmentos hablan de X?" y
recuperar solo los relevantes. Es lo que permitirá, en la Fase 4, que el
Agente Redactor pida específicamente el contexto de la sección que esté
escribiendo, sin mezclar datos de otros documentos.


In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_modelo,
    persist_directory=str(Path.cwd().parent / "data" / "chroma_db"),
    collection_name="memoria_ayuntamiento",
)

print(f"Chunks indexados: {vectorstore._collection.count()}")

**Resultado: 5 chunks indexados** — coincide exactamente con los 5 chunks
generados en el paso 3.2, confirmando que ninguno se perdió ni se
duplicó durante la indexación.

**Decisión de diseño importante — persistencia en disco:** se usa
`persist_directory` para que la base vectorial sobreviva a reinicios del
notebook/kernel. Sin esto, habría que re-generar todos los embeddings
(con su coste de tiempo, aunque pequeño con solo 5 chunks) cada vez que
se retomara el trabajo. Esta carpeta se excluye de Git (`.gitignore`),
ya que son archivos binarios generados, no código fuente revisable.

**Pendiente de comprobar en el siguiente paso (3.5-3.6):** que
`vectorstore.count()` diga 5 confirma que los datos están *guardados*,
pero no confirma todavía que la *búsqueda* funcione bien — es decir, que
al preguntar algo relacionado con "convenios" se recuperen realmente los
chunks de convenios y no los del informe financiero. Eso se prueba a
continuación.

## 3.5-3.6 — Configurar el retriever y probar consultas de recuperación

**Qué se hace:** se configura el `vectorstore` como retriever (k=2, según
la decisión ya documentada de empezar con pocos resultados) y se prueban
3 preguntas de ejemplo, cada una pensada para que la respuesta correcta
esté en un documento distinto.

**Para qué:** confirmar que la búsqueda semántica funciona de verdad, no
solo que los datos están guardados (eso ya lo comprobamos en 3.4). Esto
corresponde a la evaluación manual ligera prevista en la tarea 3.8.

**En qué fijarse:** para cada pregunta, el chunk recuperado en primer
lugar debería venir del documento correcto (`fuente` en la metadata):
- "formación y empleo" → `financiero_cuatrimestral_1.pdf`
- "Cáritas" → `convenios_2026.docx`
- "colocaciones en 2026" → `agencia_colocacion.xlsx`

Si alguna pregunta recupera el documento equivocado, no seguir a la
siguiente fase sin investigarlo — significaría que el Agente Redactor
(Fase 4) podría redactar una sección con datos de otra fuente.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
preguntas_prueba = [
    "¿Cuánto se ejecutó del presupuesto de formación y empleo?",
    "¿Cuántas personas atendió el convenio con Cáritas?",
    "¿Cuántas colocaciones hubo en 2026?",
]

for pregunta in preguntas_prueba:
    print(f"=== Pregunta: {pregunta} ===")
    resultados = retriever.invoke(pregunta)
    for r in resultados:
        print(f"[{r.metadata['fuente']}]")
        print(r.page_content[:200])
        print("---")
    print()

## Hallazgo: fallo de recuperación por cabeceras compartidas entre documentos

Al evaluar las 3 preguntas de prueba, 1 de 3 falla completamente (el chunk
con la respuesta real no se recupera) y en 2 de 3 aparece un chunk de
`convenios_2026.docx` como resultado top de forma injustificada.

**Hipótesis:** los documentos ficticios comparten una cabecera casi
idéntica ("Departamento: Desarrollo Local y Empleo", "Periodo: Enero -
Abril 2026"), que en documentos tan cortos pesa desproporcionadamente en
el embedding del chunk completo, diluyendo la señal específica de cada
contenido.

**No se puede confirmar todavía si esto es:**
(a) un problema real que también aparecerá con documentos reales más
largos (donde la cabecera pesaría proporcionalmente menos), o
(b) un artefacto exclusivo del tamaño artificialmente pequeño de los
datos ficticios.

**Antes de seguir a la Fase 4, se decide investigar con un experimento
rápido** (aumentar k, y si no basta, revisar los límites del chunking)
en vez de asumir que el retriever funciona bien.

In [ ]:
resultados_k5 = vectorstore.as_retriever(search_kwargs={"k": 5}).invoke(
    "¿Cuánto se ejecutó del presupuesto de formación y empleo?"
)
for r in resultados_k5:
    print(f"[{r.metadata['fuente']}]", r.page_content[:80])

In [ ]:
resultados_con_score = vectorstore.similarity_search_with_score(
    "¿Cuánto se ejecutó del presupuesto de formación y empleo?", k=5
)
for doc, score in resultados_con_score:
    print(f"Score: {score:.4f} | [{doc.metadata['fuente']}] {doc.page_content[:60]}")

## Hallazgo confirmado: los embeddings no diferencian bien documentos cortos con vocabulario compartido

Las puntuaciones de similitud (distancia, menor = más parecido) muestran
un patrón claro: el quinto resultado (claramente irrelevante) queda bien
diferenciado (0.85), pero los otros 4 chunks —incluido el que contiene la
respuesta correcta— quedan agrupados en un rango muy estrecho (0.58-0.68).

**Causa:** los documentos ficticios comparten cabeceras institucionales
casi idénticas, que en textos tan cortos pesan más en el embedding que el
contenido específico. El sistema de búsqueda semántica pura no es fiable
en estas condiciones.

**Decisión:** en vez de intentar "arreglar" el embedding (más iteración de
parámetros con rendimientos inciertos, mismo patrón de riesgo que ya
vivimos en la Fase 2), se aprovecha algo que ya tenemos disponible y es
100% fiable: la metadata `tipo` de cada chunk, calculada de forma
determinista en el loader (Fase 1). Se combina búsqueda semántica con
filtro exacto por tipo de documento.

## La solución: combinar filtro exacto + búsqueda semántica..

In [ ]:
resultados_filtrados = vectorstore.similarity_search_with_score(
    "¿Cuánto se ejecutó del presupuesto de formación y empleo?",
    k=2,
    filter={"tipo": "financiero"},
)
for doc, score in resultados_filtrados:
    print(f"Score: {score:.4f} | [{doc.metadata['fuente']}] {doc.page_content[:100]}")

## Solución validada: filtro por metadata + búsqueda semántica

Al combinar el filtro exacto `{"tipo": "financiero"}` con la búsqueda
semántica, el chunk con la respuesta correcta aparece de forma fiable
entre los resultados — a diferencia de la búsqueda semántica pura, que
lo perdía por el ruido de vocabulario compartido entre documentos.

**Conclusión para el diseño del pipeline:** el Agente Redactor (Fase 4)
NUNCA debe hacer una búsqueda semántica "abierta" sobre toda la base
vectorial. Cada consulta debe ir acompañada del filtro `tipo` correspondiente
a la sección que se esté redactando (ej. tipo="financiero" al escribir
la sección financiera, tipo="convenio" al escribir la de convenios).

**Pendiente de confirmar con datos reales:** esta limitación de los
embeddings con documentos cortos podría atenuarse con documentos reales
más largos y con contenido más distintivo entre sí (menos cabeceras
repetidas proporcionalmente). Aun así, el filtro por `tipo` es una buena
práctica en cualquier caso — es más rápido y no depende de que el
embedding acierte, así que se mantiene la solución independientemente
de lo que muestren los datos reales.

In [ ]:
from src.rag.retriever import cargar_vectorstore, recuperar_contexto, contexto_como_texto

vectorstore_desde_modulo = cargar_vectorstore(persist_directory=str(Path.cwd().parent / "data" / "chroma_db"))

chunks = recuperar_contexto(
    vectorstore_desde_modulo,
    pregunta="¿Cuánto se ejecutó del presupuesto de formación y empleo?",
    tipo_documento="financiero",
)
print(contexto_como_texto(chunks))